In [1]:
# CELL 1: Regenerate dataset with more realistic distributions

import pandas as pd
import numpy as np
import random

np.random.seed(42)
random.seed(42)
num_records = 1000

departments = ["IT", "HR", "Finance", "Sales", "Operations"]
genders = ["Male", "Female"]
employment_types = ["Full-time", "Part-time", "Contract"]
job_levels = [1, 2, 3, 4, 5]

data = []

for i in range(num_records):
    employee_id = i + 1
    gender = random.choice(genders)
    department = random.choice(departments)
    job_level = random.choice(job_levels)
    employment_type = random.choice(employment_types)

    attendance_rate = round(np.random.uniform(0.75, 1.0), 2)
    overtime_hours = round(np.random.uniform(0, 25), 2)
    leave_days = random.randint(0, 5)
    performance_score = round(np.random.uniform(2.5, 5.0), 2)

    base_salary = job_level * random.randint(50000, 80000)
    allowances = round(base_salary * np.random.uniform(0.1, 0.25), 2)

    # NORMAL RANGE (more overlap)
    bonus = round(base_salary * np.random.uniform(0.05, 0.3), 2)
    deductions = round(base_salary * np.random.uniform(0.02, 0.2), 2)

    tax = round(base_salary * 0.1, 2)

    previous_salary = base_salary + random.randint(-10000, 10000)

    expected_salary = base_salary + allowances + bonus - tax - deductions

    # add realistic noise
    actual_salary = expected_salary + np.random.normal(0, 8000)

    data.append([
        employee_id, gender, department, job_level, employment_type,
        attendance_rate, overtime_hours, leave_days, performance_score,
        base_salary, allowances, bonus, tax, deductions,
        previous_salary, actual_salary
    ])

columns = [
    "employee_id", "gender", "department", "job_level", "employment_type",
    "attendance_rate", "overtime_hours", "leave_days", "performance_score",
    "basic_salary", "allowances", "bonus", "tax", "deductions",
    "previous_salary", "actual_salary_paid"
]

df = pd.DataFrame(data, columns=columns)

In [2]:
# CELL 2: Feature Engineering

df["expected_salary"] = (
    df["basic_salary"] + df["allowances"] + df["bonus"]
    - df["tax"] - df["deductions"]
)

df["salary_deviation"] = df["actual_salary_paid"] - df["expected_salary"]
df["salary_deviation_ratio"] = df["salary_deviation"] / df["expected_salary"]

df["bonus_ratio"] = df["bonus"] / df["basic_salary"]
df["deduction_ratio"] = df["deductions"] / df["basic_salary"]

df["salary_change"] = df["actual_salary_paid"] - df["previous_salary"]

In [3]:

# CELL 3: Inject more diverse anomalies (HIGH + LOW)

num_anomalies = int(0.08 * num_records)
anomaly_indices = np.random.choice(df.index, num_anomalies, replace=False)

for idx in anomaly_indices:

    anomaly_type = random.choice([
        "high_bonus",
        "high_deduction",
        "salary_jump",
        "low_bonus",        # NEW
        "low_deduction"     # NEW
    ])

    if anomaly_type == "high_bonus":
        df.loc[idx, "bonus"] = df.loc[idx, "basic_salary"] * np.random.uniform(1.2, 1.8)

    elif anomaly_type == "high_deduction":
        df.loc[idx, "deductions"] = df.loc[idx, "basic_salary"] * np.random.uniform(0.3, 0.5)

    elif anomaly_type == "salary_jump":
        df.loc[idx, "actual_salary_paid"] *= np.random.uniform(1.2, 1.5)

    # NEW CASES
    elif anomaly_type == "low_bonus":
        df.loc[idx, "bonus"] = df.loc[idx, "basic_salary"] * np.random.uniform(0.001, 0.02)

    elif anomaly_type == "low_deduction":
        df.loc[idx, "deductions"] = df.loc[idx, "basic_salary"] * np.random.uniform(0.001, 0.01)

In [4]:
# CELL 4: Recompute features after anomaly injection

df["expected_salary"] = (
    df["basic_salary"] + df["allowances"] + df["bonus"]
    - df["tax"] - df["deductions"]
)

df["salary_deviation"] = df["actual_salary_paid"] - df["expected_salary"]
df["salary_deviation_ratio"] = df["salary_deviation"] / df["expected_salary"]

df["bonus_ratio"] = df["bonus"] / df["basic_salary"]
df["deduction_ratio"] = df["deductions"] / df["basic_salary"]

df["salary_change"] = df["actual_salary_paid"] - df["previous_salary"]

In [5]:
# CELL 5: Create softer anomaly labels

df["anomaly_label"] = 0

for i in df.index:
    prob = np.random.rand()

    if (
        (df.loc[i, "salary_deviation_ratio"] > 0.3 and prob > 0.3) or
        (df.loc[i, "bonus_ratio"] > 1.2 and prob > 0.4) or
        (df.loc[i, "deduction_ratio"] > 0.4 and prob > 0.5)
    ):
        df.loc[i, "anomaly_label"] = 1

In [6]:
# CELL 6: Add label noise (simulate real-world imperfection)

noise_ratio = 0.05
num_noisy = int(noise_ratio * len(df))

noisy_indices = np.random.choice(df.index, num_noisy, replace=False)

df.loc[noisy_indices, "anomaly_label"] = 1 - df.loc[noisy_indices, "anomaly_label"]

In [7]:
# CELL 7: Save dataset

df.to_csv("payroll_dataset_fixed.csv", index=False)

In [8]:
# CELL 8: Check anomaly distribution

print(df["anomaly_label"].value_counts())

anomaly_label
0    934
1     66
Name: count, dtype: int64
